# 🎼 SoundStaff 고품질 변환

음원(mp3 등)에서 **반주를 지우고 MIDI 파일로** 바꿔 줍니다. 그 파일을 SoundStaff에 놓으면 악보가 됩니다.

## 사용 방법
1. 아래 **▶ 버튼을 누릅니다.** (코드는 몰라도 됩니다. 접혀 있어서 보이지 않아요.)
2. **[파일 선택]** 이 나오면 음원 파일을 고릅니다.
3. 몇 분 기다리면 **MIDI 파일이 자동으로 내려받아집니다.**
4. 이 창을 닫고 SoundStaff 화면에 그 파일을 놓습니다.

> 💡 위쪽 [런타임] → [런타임 유형 변경]에서 **GPU**를 고르고 시작하면 더 빠릅니다. (안 골라도 됩니다)

In [ ]:
#@title ▶ 이 버튼 하나만 누르세요 { display-mode: "form" }
#@markdown 누르면 **[파일 선택]** 이 나옵니다. 음원 파일을 고르고 기다리면, 끝나는 대로 **MIDI 파일이 자동으로 내려받아집니다.**
REMOVE_ACCOMPANIMENT = True  #@param {type:"boolean"}
#@markdown ↑ 체크: 반주를 지우고 목소리(멜로디)만 변환합니다. 연주곡이면 체크를 해제하세요.

import glob, os, re, sys, time
from google.colab import files

def step(msg):
    print("\n" + "=" * 8, msg, "=" * 8)

try:
    step("1/4  준비하는 중 (처음 한 번 1~3분)")
    !pip install -q -U demucs basic-pitch

    step("2/4  음원 파일을 골라 주세요")
    os.makedirs("/content/input", exist_ok=True)
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("파일이 선택되지 않았습니다. 다시 ▶를 눌러 파일을 골라 주세요.")
    orig_name = list(uploaded.keys())[0]
    base_name, ext = os.path.splitext(orig_name)
    tmp_path = "/content/input/track" + ext.lower()
    os.replace(orig_name, tmp_path)
    print("올린 파일:", orig_name)

    source_file = tmp_path
    if REMOVE_ACCOMPANIMENT:
        step("3/4  반주를 지우는 중 (곡 길이에 따라 1~5분)")
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
        if device == "cpu":
            print("※ GPU가 꺼져 있어 조금 오래 걸립니다. (위쪽 [런타임] → [런타임 유형 변경]에서 GPU를 고르면 빨라집니다)")
        !python -m demucs --two-stems=vocals --mp3 --mp3-bitrate 320 -n htdemucs -d {device} -o /content/separated {tmp_path}
        vocal_files = glob.glob("/content/separated/htdemucs/track/vocals.*")
        if not vocal_files:
            raise RuntimeError("반주를 지우지 못했습니다. ▶를 다시 눌러 보세요. (그래도 안 되면 위 체크를 해제하고 다시 시도)")
        source_file = vocal_files[0]
    else:
        step("3/4  반주 지우기는 건너뜁니다")

    step("4/4  MIDI로 바꾸는 중 (몇십 초)")
    from basic_pitch.inference import predict_and_save
    from basic_pitch import ICASSP_2022_MODEL_PATH
    os.makedirs("/content/midi", exist_ok=True)
    predict_and_save([source_file], "/content/midi", save_midi=True, sonify_midi=False,
                     save_model_outputs=False, save_notes=False, model_or_model_path=ICASSP_2022_MODEL_PATH)
    midi_files = glob.glob("/content/midi/*.mid")
    if not midi_files:
        raise RuntimeError("MIDI를 만들지 못했습니다. 소리가 너무 작거나 비어 있는 파일일 수 있습니다.")

    dl_name = re.sub(r'[\\/:*?"<>|]+', "_", base_name) + ".mid"
    final_path = "/content/midi/" + dl_name
    os.replace(midi_files[0], final_path)
    files.download(final_path)
    print("\n✅ 끝났습니다!  '" + dl_name + "' 파일이 내려받아집니다.")
    print("이제 이 Colab 창을 닫고, SoundStaff 화면에 그 파일을 놓으세요.")
except RuntimeError as e:
    print("\n❌", e)
except Exception as e:
    print("\n❌ 문제가 생겼습니다:", type(e).__name__, "-", e)
    print("   [런타임] → [런타임 다시 시작]을 누른 뒤 ▶를 다시 눌러 보세요. 계속되면 이 화면을 캡처해서 알려 주세요.")

---
### 참고
- 올린 음원은 **내 구글 계정의 Colab에만** 올라가며 SoundStaff로 전송되지 않습니다. 끝나면 이 창을 닫으면 됩니다.
- 사용 권한이 있는 음원만 올려 주세요. 자동 분석이라 틀린 음이 있을 수 있어요. 악보 프로그램에서 고쳐 주세요.
- 무료 Colab은 사용 시간 제한이 있어 이용이 몰리면 잠시 막힐 수 있습니다. 그럴 땐 나중에 다시 해 보세요.
- 파일이 안 내려받아지면 브라우저의 팝업 차단을 풀거나, 왼쪽 폴더 아이콘 → `midi` 폴더에서 직접 받으세요.
- 사용 도구: Demucs(반주 분리, MIT), Basic Pitch(음표 찾기, Apache-2.0)